# Random Forest - Sweeps

This notebook demonstrates how to use the Random Forest pipeline for predicting match outcomes between two teams, following the same structure as the XGBoost model.

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import dotenv

import wandb
from src.api.run import sweep_randomforest_ensemble
from src.api.sweep import wandb_sweep

D:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statemen

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: schurtenberger-david (david-schurtenberger) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiments

In [3]:
max_runs = 100
sweep_config = {
    "name": "Random Forest Ensemble",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "randomforest_config": {
            "parameters": {
                "n_estimators": {"distribution": "int_uniform", "min": 50, "max": 600},
                "max_depth": {"distribution": "int_uniform", "min": 5, "max": 50},
                "min_samples_split": {"distribution": "int_uniform", "min": 2, "max": 10},
                "min_samples_leaf": {"distribution": "int_uniform", "min": 1, "max": 5},
                "max_features": {"values": ["sqrt", "log2", None, 0.2, 0.5, 0.8]},
                "bootstrap": {"value": True},
                "max_samples": {"distribution": "uniform", "min": 0.2, "max": 0.8},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_randomforest_ensemble, run_count=max_runs, project="random-forest")

## Submission from Best Model

In [3]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.experiments.config import RunConfig
from src.models.randomforest import EnsembleRandomForestRegressorModel, RandomForestHyperparamConfig
from src.submissions import create_submission

In [4]:
run = wandb.Api().run("aicomp-mmlm/random-forest/34eylyhs")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 16,
  'start_season': 2003,
  'valid_season': 2025},
 'randomforest_config': {'bootstrap': True,
  'max_depth': 6,
  'max_samples': 0.6516785467447809,
  'max_features': 'log2',
  'n_estimators': 566,
  'min_samples_leaf': 5,
  'min_samples_split': 10}}

In [5]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=16, valid_season=2025, start_season=2003, data_loader='season_average_ensemble')

In [6]:
hyperparameters = RandomForestHyperparamConfig(**config.get("randomforest_config", {}))
hyperparameters

RandomForestHyperparamConfig(n_estimators=566, max_depth=6, min_samples_split=10, min_samples_leaf=5, max_features='log2', bootstrap=True, max_samples=0.6516785467447809, min_impurity_decrease=0.0, max_leaf_nodes=None, min_weight_fraction_leaf=0.0, random_state=42, n_jobs=-1, verbose=0, warm_start=False, ccp_alpha=0.0)

In [7]:
model = EnsembleRandomForestRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [8]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_random_forest_ensemble_{season}.csv", fit=True)

metrics: {'train_brier_ensemble': np.float64(0.14707288730967347)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.17649583167716176)}, step: 2003
metrics: {'train_brier_ensemble': np.float64(0.14701819761185206)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.1800527143686036)}, step: 2004
metrics: {'train_brier_ensemble': np.float64(0.14735501215937669)}, step: 2005
metrics: {'valid_brier_ensemble': np.float64(0.17588390134997556)}, step: 2005
metrics: {'train_brier_ensemble': np.float64(0.14674370394206468)}, step: 2006
metrics: {'valid_brier_ensemble': np.float64(0.19648185529193057)}, step: 2006
metrics: {'train_brier_ensemble': np.float64(0.1478214396179189)}, step: 2007
metrics: {'valid_brier_ensemble': np.float64(0.1605542818236191)}, step: 2007
metrics: {'train_brier_ensemble': np.float64(0.1483557774203348)}, step: 2008
metrics: {'valid_brier_ensemble': np.float64(0.15283920521817723)}, step: 2008
metrics: {'train_brier_ensemble': np.float64(0.147648843

WindowsPath('D:/git/code/submissions/submission_random_forest_ensemble_2025.csv')

## Sweep with Default Features

In [3]:
max_runs = 100
sweep_config = {
    "name": "Random Forest Ensemble (Default Features)",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "randomforest_config": {
            "parameters": {
                "n_estimators": {"distribution": "int_uniform", "min": 50, "max": 600},
                "max_depth": {"distribution": "int_uniform", "min": 5, "max": 50},
                "min_samples_split": {"distribution": "int_uniform", "min": 2, "max": 10},
                "min_samples_leaf": {"distribution": "int_uniform", "min": 1, "max": 5},
                "max_features": {"values": ["sqrt", "log2", None, 0.2, 0.5, 0.8]},
                "bootstrap": {"value": True},
                "max_samples": {"distribution": "uniform", "min": 0.2, "max": 0.8},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"value": 0},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_randomforest_ensemble, run_count=max_runs, project="random-forest")

### Submission from Best Model

In [9]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.experiments.config import RunConfig
from src.models.randomforest import EnsembleRandomForestRegressorModel, RandomForestHyperparamConfig
from src.submissions import create_submission

In [10]:
run = wandb.Api().run("aicomp-mmlm/random-forest/wbtbxxuy")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 0,
  'start_season': 2003,
  'valid_season': 2025},
 'randomforest_config': {'bootstrap': True,
  'max_depth': 5,
  'max_samples': 0.36272370485247096,
  'max_features': 0.5,
  'n_estimators': 597,
  'min_samples_leaf': 5,
  'min_samples_split': 7}}

In [11]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=0, valid_season=2025, start_season=2003, data_loader='season_average_ensemble')

In [12]:
hyperparameters = RandomForestHyperparamConfig(**config.get("randomforest_config", {}))
hyperparameters

RandomForestHyperparamConfig(n_estimators=597, max_depth=5, min_samples_split=7, min_samples_leaf=5, max_features=0.5, bootstrap=True, max_samples=0.36272370485247096, min_impurity_decrease=0.0, max_leaf_nodes=None, min_weight_fraction_leaf=0.0, random_state=42, n_jobs=-1, verbose=0, warm_start=False, ccp_alpha=0.0)

In [13]:
model = EnsembleRandomForestRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [14]:
season = 2025
create_submission(
    season=season, model=model, filename=f"submission_random_forest_ensemble_default_features_{season}.csv", fit=True
)

metrics: {'train_brier_ensemble': np.float64(0.15290291797867506)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.1780012094528846)}, step: 2003
metrics: {'train_brier_ensemble': np.float64(0.1530831943392926)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.17396877071524108)}, step: 2004
metrics: {'train_brier_ensemble': np.float64(0.15323360433667188)}, step: 2005
metrics: {'valid_brier_ensemble': np.float64(0.16964515385049633)}, step: 2005
metrics: {'train_brier_ensemble': np.float64(0.152515018335689)}, step: 2006
metrics: {'valid_brier_ensemble': np.float64(0.19662977620106661)}, step: 2006
metrics: {'train_brier_ensemble': np.float64(0.15359101333862266)}, step: 2007
metrics: {'valid_brier_ensemble': np.float64(0.1600634457450294)}, step: 2007
metrics: {'train_brier_ensemble': np.float64(0.1535855649522515)}, step: 2008
metrics: {'valid_brier_ensemble': np.float64(0.1587251565162442)}, step: 2008
metrics: {'train_brier_ensemble': np.float64(0.153210158069

WindowsPath('D:/git/code/submissions/submission_random_forest_ensemble_default_features_2025.csv')